The dataset class is adapted from the [UvA Deep Learning tutorial 2](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/JAX/tutorial2/Introduction_to_JAX.html), which is released under the [MIT License](https://github.com/phlippe/uvadlc_notebooks/blob/master/LICENSE.md) (Copyright (c) 2020 Phillip Lippe).

In [ ]:
import numpy as np
import torch
import torch.utils.data as data
from itertools import combinations

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()

from src.pipelines.unified_training import unified_train_model
from src.models.create_model import create_model
from src.data.dataset import choice_collate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def extract_embeddings(model, model_name, num_items, num_users, num_prior_samples=50):
    model.eval()
    model_device = next(model.parameters()).device

    with torch.no_grad():
        item_embs = model.item_embs.detach().cpu().numpy()

        if model_name == "MNL":
            user_embs = model.user_mean.detach().cpu().numpy()
        else:
            user_samples = model.draw_user_embedding_sample(
                user_ids=torch.arange(num_users, device=model_device),
                num_samples=num_users,
                num_prior_samples=num_prior_samples,
            ).detach().cpu().numpy()

            user_embs = {
                "samples": user_samples,
                "mean": np.mean(user_samples, axis=1),
                "std": np.std(user_samples, axis=1),
            }

    return item_embs, user_embs


def plot_1d_embeddings(item_embs, user_embs, model_name, italian_ids, chinese_ids, no_choice_id, padding_id, ax=None):
    plt.rcParams['font.family'] = 'Georgia'

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))

    out_dim = item_embs.shape[1]
    if out_dim > 1:
        print(f"Warning: Embeddings have {out_dim} dimensions. Only plotting first dimension.")

    item_vals = item_embs[:, 0] if out_dim > 1 else item_embs.flatten()

    num_items = len(item_vals)
    y_positions = {}

    y = 0

    y_positions[no_choice_id] = y
    y += 0.2

    for item_id in chinese_ids:
        if item_id < len(item_vals):
            y_positions[item_id] = y
            y += 0.2

    for item_id in italian_ids:
        if item_id < len(item_vals):
            y_positions[item_id] = y
            y += 0.2

    user_y = -0.25

    colors = {'italian': '#e74c3c', 'chinese': '#3498db', 'no_choice': '#7f8c8d', 'padding': '#bdc3c7'}

    chinese_display_num = {cid: i + 1 for i, cid in enumerate(chinese_ids)}

    is_mxl = model_name == "MXL"

    for item_id, y_pos in y_positions.items():
        if item_id >= len(item_vals):
            continue
        x_val = item_vals[item_id]

        if item_id == no_choice_id:
            color = colors['no_choice']
            label = "No-Choice"
            marker = 'o'
            if is_mxl:
                text_offset = (10, 0)
                text_ha = 'left'
            else:
                text_offset = (-10, 0)
                text_ha = 'right'
        elif item_id in italian_ids:
            color = colors['italian']
            label = f"Italian-{item_id}"
            marker = 'o'
            if is_mxl:
                text_offset = (-10, 0)
                text_ha = 'right'
            else:
                text_offset = (10, 0)
                text_ha = 'left'
        elif item_id in chinese_ids:
            color = colors['chinese']
            label = f"Chinese-{chinese_display_num[item_id]}"
            marker = 'o'
            if is_mxl:
                text_offset = (10, 0)
                text_ha = 'left'
            else:
                text_offset = (-10, 0)
                text_ha = 'right'
        else:
            color = colors['padding']
            label = f"Item-{item_id}"
            marker = 'o'
            text_offset = (10, 0)
            text_ha = 'left'

        ax.scatter(x_val, y_pos, c=color, s=80, marker=marker, edgecolors='black', linewidths=1, zorder=3)
        ax.annotate(label, (x_val, y_pos), xytext=text_offset, textcoords='offset points',
                   fontsize=10, va='center', ha=text_ha)

    if isinstance(user_embs, dict):
        user_mean = user_embs["mean"]
        user_samples = user_embs["samples"]
        user_std = user_embs["std"]

        if user_mean.ndim > 1:
            user_mean = user_mean[0]
            user_samples = user_samples[0]
            user_std = user_std[0]

        user_mean_val = float(user_mean[0]) if hasattr(user_mean, '__len__') else float(user_mean)

        sample_vals = user_samples[:, 0] if user_samples.ndim > 1 else user_samples
        ax.scatter(sample_vals, [user_y] * len(sample_vals), c='purple', s=12, alpha=0.3, zorder=2)

        ax.annotate("User",
                   (user_mean_val, user_y), xytext=(-5, 5), textcoords='offset points',
                   fontsize=10, va='bottom', ha='right', fontweight='bold', color='purple')
    else:
        if user_embs.ndim > 1:
            user_val = float(user_embs[0, 0])
        else:
            user_val = float(user_embs[0])

        ax.scatter(user_val, user_y, c='purple', s=100, marker='o', edgecolors='black', linewidths=1, zorder=4)
        ax.annotate("User",
                   (user_val, user_y), xytext=(5, 5), textcoords='offset points',
                   fontsize=10, va='bottom', ha='left', fontweight='bold', color='purple')

    ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3, linewidth=1)

    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_yticks([])
    ax.set_title(f'{model_name}', fontsize=14)

    ax.spines['bottom'].set_visible(False)

    ax.grid(axis='x', alpha=0.3)

    return ax

In [ ]:
MASTER_SEED = 42

seed = MASTER_SEED
torch.manual_seed(seed)
np.random.seed(seed)

NUM_ITALIAN = 2
NUM_CHINESE = 5
NUM_RESTAURANTS = NUM_ITALIAN + NUM_CHINESE
NUM_ITEMS = NUM_RESTAURANTS + 1
NUM_USERS = 1
NO_CHOICE_ID = 0
PADDING_ID = NUM_ITEMS
SLATE_SIZE = 5
NO_CHOICE_PROBABILITY = 0.0

ITALIAN_IDS = list(range(1, NUM_ITALIAN + 1))
CHINESE_IDS = list(range(NUM_ITALIAN + 1, NUM_RESTAURANTS + 1))

print(f"Master seed: {MASTER_SEED}")
print(f"Italian restaurant IDs: {ITALIAN_IDS}")
print(f"Chinese restaurant IDs: {CHINESE_IDS}")
print(f"No-choice ID: {NO_CHOICE_ID}")
print(f"Padding ID: {PADDING_ID}")
print(f"Slate size: {SLATE_SIZE} restaurants + no-choice")

In [ ]:
class NestedChoiceDataset(data.Dataset):

    def __init__(self, size, seed, italian_ids, chinese_ids, no_choice_id, num_items, slate_size=None, fixed_slate=None, no_choice_probability=0.5):
        super().__init__()
        self.size = size
        self.np_rng = np.random.RandomState(seed=seed)
        self.italian_ids = italian_ids
        self.chinese_ids = chinese_ids
        self.no_choice_id = no_choice_id
        self.num_items = num_items
        self.slate_size = slate_size if slate_size is not None else len(italian_ids) + len(chinese_ids)
        self.fixed_slate = fixed_slate
        self.no_choice_probability = no_choice_probability
        self.generate_choices()

    def generate_choices(self):
        all_restaurant_ids = self.italian_ids + self.chinese_ids

        if self.fixed_slate is not None:
            if self.no_choice_id not in self.fixed_slate:
                slate = [self.no_choice_id] + list(self.fixed_slate)
            else:
                slate = list(self.fixed_slate)
            self.option_ids = np.tile(slate, (self.size, 1))
        else:
            self.option_ids = np.zeros((self.size, self.slate_size + 1), dtype=int)
            self.option_ids[:, 0] = self.no_choice_id
            for i in range(self.size):
                sampled_items = self.np_rng.choice(all_restaurant_ids, size=self.slate_size, replace=False)
                self.option_ids[i, 1:] = sampled_items

        self.user_ids = np.zeros(self.size, dtype=int)
        self.choice_pos = np.zeros(self.size, dtype=int)
        self.choice_ids = np.zeros(self.size, dtype=int)

        for i in range(self.size):
            slate = self.option_ids[i]

            italian_in_slate = [idx for idx, item in enumerate(slate) if item in self.italian_ids]
            chinese_in_slate = [idx for idx, item in enumerate(slate) if item in self.chinese_ids]
            no_choice_pos = np.where(slate == self.no_choice_id)[0][0]

            if self.np_rng.random() < self.no_choice_probability:
                self.choice_pos[i] = no_choice_pos
                self.choice_ids[i] = self.no_choice_id
            else:
                if self.np_rng.random() < 0.5:
                    if len(italian_in_slate) > 0:
                        chosen_pos = self.np_rng.choice(italian_in_slate)
                        self.choice_pos[i] = chosen_pos
                        self.choice_ids[i] = slate[chosen_pos]
                    else:
                        self.choice_pos[i] = no_choice_pos
                        self.choice_ids[i] = self.no_choice_id
                else:
                    if len(chinese_in_slate) > 0:
                        chosen_pos = self.np_rng.choice(chinese_in_slate)
                        self.choice_pos[i] = chosen_pos
                        self.choice_ids[i] = slate[chosen_pos]
                    else:
                        self.choice_pos[i] = no_choice_pos
                        self.choice_ids[i] = self.no_choice_id

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        user_id = self.user_ids[idx]
        option_id = self.option_ids[idx]
        choice_pos = self.choice_pos[idx]
        choice_pos_onehot = np.eye(len(option_id))[choice_pos]
        choice_id = self.choice_ids[idx]
        return (
            {"user_ids": user_id, "option_id": option_id},
            {"choice_pos": choice_pos, "choice_pos_onehot": choice_pos_onehot, "choice_id": choice_id},
        )


numpy_collate = choice_collate

In [ ]:
batch_size = 200
train_size = 2000

train_dataset = NestedChoiceDataset(
    size=train_size,
    seed=42,
    italian_ids=ITALIAN_IDS,
    chinese_ids=CHINESE_IDS,
    no_choice_id=NO_CHOICE_ID,
    num_items=NUM_ITEMS,
    slate_size=SLATE_SIZE,
    no_choice_probability=NO_CHOICE_PROBABILITY,
)
train_data_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=numpy_collate)

print("Choice distribution in training data:")
unique, counts = np.unique(train_dataset.choice_ids, return_counts=True)
for item_id, count in zip(unique, counts):
    pct = count / train_size * 100
    if item_id == NO_CHOICE_ID:
        print(f"  No-choice (id={item_id}): {pct:.1f}%")
    elif item_id in ITALIAN_IDS:
        print(f"  Italian restaurant (id={item_id}): {pct:.1f}%")
    else:
        print(f"  Chinese restaurant (id={item_id}): {pct:.1f}%")

In [ ]:
print("Data point 9:", train_dataset[9])

In [ ]:
hyperparams = {
    "learning_rate": 0.01,
    "num_early_stopping": 100,
    "out_dim": 1,
    "num_prior_samples": 32,
    "beta_max": 1.0,
    "has_intercept": False,
    "l2_weight": 0,
    "use_output_layer_norm": False,
}

In [ ]:
MNL_SEED = 1

torch.manual_seed(MNL_SEED)
mnl_model = create_model(
    model_name="MNL",
    num_users=NUM_USERS,
    num_items=NUM_ITEMS + 1,
    hyperparams=hyperparams,
    no_choice_id=NO_CHOICE_ID,
    padding_id=PADDING_ID,
    never_chosen_id=None,
).to(device)

mnl_optimizer = torch.optim.Adam(mnl_model.parameters(), lr=hyperparams["learning_rate"])

print(f"MNL model seed: {MNL_SEED}")
print(f"MNL model parameters: {sum(p.numel() for p in mnl_model.parameters()):,}")

MXL_SEED = 1

torch.manual_seed(MXL_SEED)
mxl_model = create_model(
    model_name="MXL",
    num_users=NUM_USERS,
    num_items=NUM_ITEMS + 1,
    hyperparams=hyperparams,
    no_choice_id=NO_CHOICE_ID,
    padding_id=PADDING_ID,
    never_chosen_id=None,
).to(device)

mxl_optimizer = torch.optim.Adam(mxl_model.parameters(), lr=hyperparams["learning_rate"])

print(f"\nMXL model seed: {MXL_SEED}")
print(f"MXL model parameters: {sum(p.numel() for p in mxl_model.parameters()):,}")

In [ ]:
torch.manual_seed(999)
mnl_untrained_item_embs, mnl_untrained_user_embs = extract_embeddings(mnl_model, "MNL", NUM_ITEMS, NUM_USERS)
mxl_untrained_item_embs, mxl_untrained_user_embs = extract_embeddings(mxl_model, "MXL", NUM_ITEMS, NUM_USERS)

print("Untrained MNL Item embeddings shape:", mnl_untrained_item_embs.shape)
print("Untrained MNL User embeddings shape:", mnl_untrained_user_embs.shape)
print("Untrained MXL Item embeddings shape:", mxl_untrained_item_embs.shape)
if isinstance(mxl_untrained_user_embs, dict):
    print("Untrained MXL User embeddings (samples) shape:", mxl_untrained_user_embs["samples"].shape)
    print("Untrained MXL User mean:", mxl_untrained_user_embs["mean"])
    print("Untrained MXL User std:", mxl_untrained_user_embs["std"])

In [ ]:
if hyperparams["out_dim"] != 1:
    print(f"Warning: Embeddings have out_dim={hyperparams['out_dim']}, expected 1 for 1D plotting.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    plot_1d_embeddings(mnl_untrained_item_embs, mnl_untrained_user_embs, "MNL (Untrained)",
                      ITALIAN_IDS, CHINESE_IDS, NO_CHOICE_ID, PADDING_ID, ax=axes[0])
    plot_1d_embeddings(mxl_untrained_item_embs, mxl_untrained_user_embs, "MXL (Untrained)",
                      ITALIAN_IDS, CHINESE_IDS, NO_CHOICE_ID, PADDING_ID, ax=axes[1])

    plt.tight_layout()
    plt.show()

    print("\n" + "="*60)
    print("UNTRAINED EMBEDDING VALUES")
    print("="*60)
    print("\nUntrained MNL Model:")
    print(f"  User embedding: {mnl_untrained_user_embs.flatten()}")
    print(f"  Item embeddings:")
    for i, emb in enumerate(mnl_untrained_item_embs):
        if i == NO_CHOICE_ID:
            print(f"    No-Choice (id={i}): {emb}")
        elif i in ITALIAN_IDS:
            print(f"    Italian-{i}: {emb}")
        elif i in CHINESE_IDS:
            print(f"    Chinese-{i}: {emb}")

    print("\nUntrained MXL Model:")
    if isinstance(mxl_untrained_user_embs, dict):
        print(f"  User mean: {mxl_untrained_user_embs['mean']}")
        print(f"  User std: {mxl_untrained_user_embs['std']}")
    print(f"  Item embeddings:")
    for i, emb in enumerate(mxl_untrained_item_embs):
        if i == NO_CHOICE_ID:
            print(f"    No-Choice (id={i}): {emb}")
        elif i in ITALIAN_IDS:
            print(f"    Italian-{i}: {emb}")
        elif i in CHINESE_IDS:
            print(f"    Chinese-{i}: {emb}")

In [ ]:
torch.manual_seed(MXL_SEED + 1)
trained_mxl_model, mxl_stats = unified_train_model(
    model=mxl_model,
    model_name="MXL",
    optimizer=mxl_optimizer,
    train_data_loader=train_data_loader,
    val_data_loader=None,
    num_epochs=2001,
    num_early_stopping=hyperparams["num_early_stopping"],
    results_dir="data/results/restaurant_example",
    device=device,
)
print("MXL training complete!")

In [ ]:
torch.manual_seed(MNL_SEED + 1)
trained_mnl_model, mnl_stats = unified_train_model(
    model=mnl_model,
    model_name="MNL",
    optimizer=mnl_optimizer,
    train_data_loader=train_data_loader,
    val_data_loader=None,
    num_epochs=2001,
    num_early_stopping=hyperparams["num_early_stopping"],
    results_dir="data/results/restaurant_example",
    device=device,
)
print("MNL training complete!")

In [ ]:
torch.manual_seed(999)
mnl_item_embs, mnl_user_embs = extract_embeddings(trained_mnl_model, "MNL", NUM_ITEMS, NUM_USERS)
mxl_item_embs, mxl_user_embs = extract_embeddings(trained_mxl_model, "MXL", NUM_ITEMS, NUM_USERS)

print("Trained MNL Item embeddings shape:", mnl_item_embs.shape)
print("Trained MNL User embeddings shape:", mnl_user_embs.shape)
print("Trained MXL Item embeddings shape:", mxl_item_embs.shape)
if isinstance(mxl_user_embs, dict):
    print("Trained MXL User embeddings (samples) shape:", mxl_user_embs["samples"].shape)
    print("Trained MXL User mean:", mxl_user_embs["mean"])
    print("Trained MXL User std:", mxl_user_embs["std"])

In [ ]:
if hyperparams["out_dim"] != 1:
    print(f"Warning: Embeddings have out_dim={hyperparams['out_dim']}, expected 1 for 1D plotting.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(6, 2.2))

    plot_1d_embeddings(mnl_item_embs, mnl_user_embs, "MNL", ITALIAN_IDS, CHINESE_IDS, NO_CHOICE_ID, PADDING_ID, ax=axes[0])
    plot_1d_embeddings(mxl_item_embs, mxl_user_embs, "MXL", ITALIAN_IDS, CHINESE_IDS, NO_CHOICE_ID, PADDING_ID, ax=axes[1])

    axes[0].set_title("")
    axes[1].set_title("")
    axes[0].text(0.94, 0.98, "MNL", transform=axes[0].transAxes, fontsize=12, fontweight='bold', va='top', ha='right')
    axes[1].text(0.06, 0.98, "MXL", transform=axes[1].transAxes, fontsize=12, fontweight='bold', va='top', ha='left')


    for ax in axes:
        ax.tick_params(axis='x', pad=-2)

    plt.tight_layout(pad=0.3)
    plt.show()

    print("\n" + "="*60)
    print("EMBEDDING VALUES")
    print("="*60)
    print("\nMNL Model:")
    print(f"  User embedding: {mnl_user_embs.flatten()}")
    print(f"  Item embeddings:")
    for i, emb in enumerate(mnl_item_embs):
        if i == NO_CHOICE_ID:
            print(f"    No-Choice (id={i}): {emb}")
        elif i in ITALIAN_IDS:
            print(f"    Italian-{i}: {emb}")
        elif i in CHINESE_IDS:
            print(f"    Chinese-{i}: {emb}")

    print("\nMXL Model:")
    if isinstance(mxl_user_embs, dict):
        print(f"  User mean: {mxl_user_embs['mean']}")
        print(f"  User std: {mxl_user_embs['std']}")
    print(f"  Item embeddings:")
    for i, emb in enumerate(mxl_item_embs):
        if i == NO_CHOICE_ID:
            print(f"    No-Choice (id={i}): {emb}")
        elif i in ITALIAN_IDS:
            print(f"    Italian-{i}: {emb}")
        elif i in CHINESE_IDS:
            print(f"    Chinese-{i}: {emb}")

    fig.savefig(
        "data/plots/figure_restaurant_example_embs.pdf",
        bbox_inches="tight",
        dpi=100
    )

In [ ]:
all_restaurant_ids = ITALIAN_IDS + CHINESE_IDS
all_2_item_combos = list(combinations(all_restaurant_ids, 2))

print(f"Number of possible 2-item slates: {len(all_2_item_combos)}")

slate_size = len(train_dataset.option_ids[0])

all_slates = []
for combo in all_2_item_combos:
    slate = [NO_CHOICE_ID, combo[0], combo[1]]
    while len(slate) < slate_size:
        slate.append(PADDING_ID)
    all_slates.append(slate)

all_slates = np.array(all_slates)
print(f"Slate shape: {all_slates.shape}")
print(f"Example slates:")
for i in range(min(5, len(all_slates))):
    items = all_slates[i]
    item_types = []
    for item in items:
        if item == NO_CHOICE_ID:
            item_types.append("NC")
        elif item in ITALIAN_IDS:
            item_types.append(f"I{item}")
        elif item in CHINESE_IDS:
            item_types.append(f"C{item}")
        else:
            item_types.append("PAD")
    print(f"  {items} -> {item_types}")

In [ ]:
def get_no_choice_probs(model, model_name, slates, no_choice_pos=0, num_mc_samples=100, seed=12345):
    num_slates = len(slates)
    model_device = next(model.parameters()).device

    input_batch = {
        "user_ids": torch.zeros(num_slates, dtype=torch.long, device=model_device),
        "option_id": torch.as_tensor(np.asarray(slates), dtype=torch.long, device=model_device),
    }

    torch.manual_seed(seed)
    model.eval()

    with torch.no_grad():
        if model_name == "MNL":
            logits, probs = model(input_batch)
            no_choice_probs = probs[:, no_choice_pos]
        else:
            all_probs = []
            for i in range(num_mc_samples):
                logits, probs = model(input_batch)
                all_probs.append(probs[:, no_choice_pos])
            no_choice_probs = torch.stack(all_probs).mean(dim=0)

    return no_choice_probs.detach().cpu().numpy()

In [ ]:
mnl_no_choice_probs = get_no_choice_probs(trained_mnl_model, "MNL", all_slates, seed=300)

mxl_no_choice_probs = get_no_choice_probs(trained_mxl_model, "MXL", all_slates, num_mc_samples=200, seed=301)

mnl_best_idx = np.argmin(mnl_no_choice_probs)
mxl_best_idx = np.argmin(mxl_no_choice_probs)

mnl_best_slate = all_slates[mnl_best_idx]
mxl_best_slate = all_slates[mxl_best_idx]

def slate_to_str(slate):
    items = []
    for item in slate:
        if item == NO_CHOICE_ID:
            items.append("NoChoice")
        elif item in ITALIAN_IDS:
            items.append(f"Italian-{item}")
        elif item in CHINESE_IDS:
            items.append(f"Chinese-{item}")
        elif item == PADDING_ID:
            continue
        else:
            items.append(f"Item-{item}")
    return items

print("=" * 60)
print("OPTIMAL SLATES COMPARISON")
print("=" * 60)
print(f"\nMNL Best Slate: {slate_to_str(mnl_best_slate)}")
print(f"  Predicted no-choice probability: {mnl_no_choice_probs[mnl_best_idx]:.4f}")
print(f"  Predicted choice probability: {1 - mnl_no_choice_probs[mnl_best_idx]:.4f}")

print(f"\nMXL Best Slate: {slate_to_str(mxl_best_slate)}")
print(f"  Predicted no-choice probability: {mxl_no_choice_probs[mxl_best_idx]:.4f}")
print(f"  Predicted choice probability: {1 - mxl_no_choice_probs[mxl_best_idx]:.4f}")

In [ ]:
def categorize_slate(slate):
    items = [item for item in slate if item != NO_CHOICE_ID and item != PADDING_ID]
    num_italian = sum(1 for item in items if item in ITALIAN_IDS)
    num_chinese = sum(1 for item in items if item in CHINESE_IDS)

    if num_italian == 2:
        return "2 Italian"
    elif num_chinese == 2:
        return "2 Chinese"
    else:
        return "1 Italian + 1 Chinese"

slate_categories = [categorize_slate(slate) for slate in all_slates]

print("\nNo-choice probability by slate type:")
print("-" * 50)
for category in ["2 Italian", "2 Chinese", "1 Italian + 1 Chinese"]:
    cat_indices = [i for i, cat in enumerate(slate_categories) if cat == category]
    mnl_avg = np.mean(mnl_no_choice_probs[cat_indices])
    mxl_avg = np.mean(mxl_no_choice_probs[cat_indices])
    print(f"{category:25s} | MNL: {mnl_avg:.4f} | MXL: {mxl_avg:.4f}")

print("\n" + "-" * 50)
print(f"MNL optimal slate type: {categorize_slate(mnl_best_slate)}")
print(f"MXL optimal slate type: {categorize_slate(mxl_best_slate)}")

In [ ]:
def simulate_true_choice_rate(slate, italian_ids, chinese_ids, no_choice_id, num_simulations=100000, seed=None):
    np_rng = np.random.RandomState(seed=seed)

    italian_in_slate = [item for item in slate if item in italian_ids]
    chinese_in_slate = [item for item in slate if item in chinese_ids]

    num_choices = 0

    for _ in range(num_simulations):
        if np_rng.random() < NO_CHOICE_PROBABILITY:
            continue

        if np_rng.random() < 0.5:
            if len(italian_in_slate) > 0:
                num_choices += 1
        else:
            if len(chinese_in_slate) > 0:
                num_choices += 1

    choice_rate = num_choices / num_simulations
    return choice_rate


SIM_SEED_BASE = 500

mnl_slate_items = [item for item in mnl_best_slate if item != PADDING_ID]
mnl_true_choice_rate = simulate_true_choice_rate(
    mnl_slate_items, ITALIAN_IDS, CHINESE_IDS, NO_CHOICE_ID, seed=SIM_SEED_BASE
)

mxl_slate_items = [item for item in mxl_best_slate if item != PADDING_ID]
mxl_true_choice_rate = simulate_true_choice_rate(
    mxl_slate_items, ITALIAN_IDS, CHINESE_IDS, NO_CHOICE_ID, seed=SIM_SEED_BASE + 1
)

optimal_slate_example = [NO_CHOICE_ID, ITALIAN_IDS[0], CHINESE_IDS[0]]
optimal_true_choice_rate = simulate_true_choice_rate(
    optimal_slate_example, ITALIAN_IDS, CHINESE_IDS, NO_CHOICE_ID, seed=SIM_SEED_BASE + 2
)

print("=" * 60)
print("SIMULATED TRUE PERFORMANCE (100,000 simulations)")
print("=" * 60)
print(f"\nMNL Optimal Slate: {slate_to_str(mnl_best_slate)}")
print(f"  Slate type: {categorize_slate(mnl_best_slate)}")
print(f"  TRUE choice rate: {mnl_true_choice_rate:.2%}")
print(f"  TRUE no-choice rate: {1 - mnl_true_choice_rate:.2%}")

print(f"\nMXL Optimal Slate: {slate_to_str(mxl_best_slate)}")
print(f"  Slate type: {categorize_slate(mxl_best_slate)}")
print(f"  TRUE choice rate: {mxl_true_choice_rate:.2%}")
print(f"  TRUE no-choice rate: {1 - mxl_true_choice_rate:.2%}")

print(f"\nTheoretical Optimal (1 Italian + 1 Chinese):")
print(f"  TRUE choice rate: {optimal_true_choice_rate:.2%}")

In [ ]:
import pandas as pd

summary_data = {
    "Model": ["MNL", "MXL", "Theoretical Optimal"],
    "Optimal Slate": [
        categorize_slate(mnl_best_slate),
        categorize_slate(mxl_best_slate),
        "1 Italian + 1 Chinese"
    ],
    "Predicted Choice Rate": [
        f"{(1 - mnl_no_choice_probs[mnl_best_idx]):.1%}",
        f"{(1 - mxl_no_choice_probs[mxl_best_idx]):.1%}",
        "-"
    ],
    "TRUE Choice Rate": [
        f"{mnl_true_choice_rate:.1%}",
        f"{mxl_true_choice_rate:.1%}",
        f"{optimal_true_choice_rate:.1%}"
    ],
}

df = pd.DataFrame(summary_data)
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(df.to_string(index=False))
print("=" * 70)

print("\nKEY INSIGHT:")
print("-" * 70)
if categorize_slate(mnl_best_slate) != "1 Italian + 1 Chinese" and categorize_slate(mxl_best_slate) == "1 Italian + 1 Chinese":
    print("✓ The MNL incorrectly recommends a slate with duplicate cuisines")
    print("✓ The MXL correctly recommends a diverse slate (1 Italian + 1 Chinese)")
    print(f"✓ The MXL slate achieves {mxl_true_choice_rate:.1%} choice rate vs MNL's {mnl_true_choice_rate:.1%}")
    print(f"✓ This is a {(mxl_true_choice_rate - mnl_true_choice_rate) / mnl_true_choice_rate:.0%} improvement!")
elif categorize_slate(mnl_best_slate) == categorize_slate(mxl_best_slate):
    print("Both models recommend similar slates in this run.")
    print("The difference may emerge with different random seeds or hyperparameters.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

categories = ["2 Italian", "2 Chinese", "1 Italian + 1 Chinese"]
true_no_choice_rates = []

for idx, cat in enumerate(categories):
    if cat == "2 Italian":
        example_slate = [NO_CHOICE_ID, ITALIAN_IDS[0], ITALIAN_IDS[1]]
    elif cat == "2 Chinese":
        example_slate = [NO_CHOICE_ID, CHINESE_IDS[0], CHINESE_IDS[1]]
    else:
        example_slate = [NO_CHOICE_ID, ITALIAN_IDS[0], CHINESE_IDS[0]]
    rate = 1 - simulate_true_choice_rate(example_slate, ITALIAN_IDS, CHINESE_IDS, NO_CHOICE_ID, seed=600 + idx)
    true_no_choice_rates.append(rate)

mnl_cat_probs = [np.mean(mnl_no_choice_probs[[i for i, c in enumerate(slate_categories) if c == cat]]) for cat in categories]
mxl_cat_probs = [np.mean(mxl_no_choice_probs[[i for i, c in enumerate(slate_categories) if c == cat]]) for cat in categories]

x = np.arange(len(categories))
width = 0.25

ax1 = axes[0]
bars1 = ax1.bar(x - width, mnl_cat_probs, width, label='MNL Predicted', color='steelblue', alpha=0.8)
bars2 = ax1.bar(x, mxl_cat_probs, width, label='MXL Predicted', color='darkorange', alpha=0.8)
bars3 = ax1.bar(x + width, true_no_choice_rates, width, label='True Rate', color='green', alpha=0.8)

ax1.set_ylabel('No-Choice Probability')
ax1.set_title('Predicted vs True No-Choice Probability by Slate Type')
ax1.set_xticks(x)
ax1.set_xticklabels(categories)
ax1.legend()
ax1.set_ylim(0, 1)
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
models = ['MNL\nOptimal', 'MXL\nOptimal', 'True\nOptimal']
predicted_choice = [1 - mnl_no_choice_probs[mnl_best_idx], 1 - mxl_no_choice_probs[mxl_best_idx], optimal_true_choice_rate]
true_choice = [mnl_true_choice_rate, mxl_true_choice_rate, optimal_true_choice_rate]

x2 = np.arange(len(models))
bars4 = ax2.bar(x2 - width/2, predicted_choice, width, label='Model Prediction', color='steelblue', alpha=0.8)
bars5 = ax2.bar(x2 + width/2, true_choice, width, label='True Rate', color='green', alpha=0.8)

ax2.set_ylabel('Choice Probability')
ax2.set_title('Choice Rate: Model Prediction vs Reality')
ax2.set_xticks(x2)
ax2.set_xticklabels(models)
ax2.legend()
ax2.set_ylim(0, 0.7)
ax2.grid(axis='y', alpha=0.3)

for bars in [bars4, bars5]:
    for bar in bars:
        height = bar.get_height()
        ax2.annotate(f'{height:.1%}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()